# Samaritan solver on an A100 (turnkey)

Serves a strong reasoning model on this Colab A100 and exposes it as an
OpenAI-compatible endpoint your **local** Samaritan harness points at —
no code change, just `SAMARITAN_URL` + `SAMARITAN_API_KEY`.

**Before running:** Runtime → Change runtime type → **A100 GPU** (Pro+).
Then Runtime → **Run all**. The last serving cell prints the two lines to
paste locally.

**Honest limits:** Colab is not 24/7 (Pro+ background ~24 h, can drop).
Good for eval pushes and self-training generation, not a deployment. The
API key protects the public tunnel URL — don't share it. Run the shutdown
cell (or stop the runtime) when done; a forgotten A100 burns compute units.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 2. Install vLLM

vLLM serves the OpenAI API natively and batches fast — right for the many
playout calls. First run pulls a lot; give it a few minutes.

In [ ]:
!pip -q install vllm huggingface_hub

## 3. Choose the model + a key

`Qwen3-30B-A3B-Thinking-2507` (FP8) is the strong reasoning model that fits
a 40 GB A100 — a large step up from the local 4B. It's a Mixture-of-Experts:
30B total parameters but only ~3B active per token, so it reasons far better
than the 4B while staying fast under vLLM. The FP8 weights are ~30 GB; the
A100 (Ampere) has no FP8 compute units, but vLLM loads FP8 **weight-only**
via its Marlin kernel automatically — weights in FP8, math in bf16, no extra
flags. **Note:** there is no `Qwen3-14B-Thinking`; the 30B-A3B MoE is the
actual next rung. If it OOMs on your runtime, lower `MAX_LEN` or switch
`MODEL` to the 4B fallback below.

In [ ]:
import secrets
MODEL = "Qwen/Qwen3-30B-A3B-Thinking-2507-FP8"   # ~30 GB FP8, fits 40 GB
# Fallbacks:
#   "Qwen/Qwen3-4B-Thinking-2507"   # tiny, always fits (same class as local — little lift)
PORT = 8000
MAX_LEN = 8192                                   # prompt + thinking trace; lower if you OOM
API_KEY = secrets.token_urlsafe(24)
print('MODEL   :', MODEL)
print('API_KEY :', API_KEY)

## 4. Start vLLM (background) and wait until it serves

`--served-model-name samaritan-playout` is the alias the harness asks for,
so nothing changes on our side. This cell returns once the model is loaded
and answering (the weight download happens here).

In [ ]:
import subprocess, time, urllib.request, json

vllm = subprocess.Popen([
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MODEL, '--served-model-name', 'samaritan-playout',
    '--api-key', API_KEY, '--port', str(PORT), '--max-model-len', str(MAX_LEN),
], stdout=open('vllm.log', 'w'), stderr=subprocess.STDOUT)

print('loading', MODEL, '(watch vllm.log for progress)...')
url = f'http://localhost:{PORT}/v1/models'
ready = False
for i in range(180):  # up to ~30 min for a cold ~30 GB pull + vLLM profiling
    if vllm.poll() is not None:
        print('vLLM exited early — tail of vllm.log:'); print(open('vllm.log').read()[-2000:]); break
    try:
        req = urllib.request.Request(url, headers={'Authorization': f'Bearer {API_KEY}'})
        if urllib.request.urlopen(req, timeout=5).status == 200:
            ready = True; break
    except Exception:
        pass
    time.sleep(10)
print('SERVING' if ready else 'not ready — check vllm.log')

## 5. Expose it with a cloudflared tunnel

Publishes the vLLM port as a public `https://…trycloudflare.com` URL (the
key is what protects it). Prints the exact two lines to paste locally.

In [ ]:
import subprocess, re, time

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
cf = subprocess.Popen(['./cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}'],
                      stdout=open('cf.log', 'w'), stderr=subprocess.STDOUT)
public = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('cf.log').read())
    if m: public = m.group(0); break
if not public:
    print('no tunnel URL yet — tail of cf.log:'); print(open('cf.log').read()[-1500:])
else:
    print('Tunnel up. Paste these into your LOCAL terminal:\n')
    print(f'  $env:SAMARITAN_URL = "{public}/v1"        # PowerShell')
    print(f'  $env:SAMARITAN_API_KEY = "{API_KEY}"')
    print(f'\n  export SAMARITAN_URL="{public}/v1"          # bash')
    print(f'  export SAMARITAN_API_KEY="{API_KEY}"')

## 6. Self-test (optional)

Proves the whole chain answers before you drive it from the laptop.

In [ ]:
import urllib.request, json
body = json.dumps({'model': 'samaritan-playout',
    'messages': [{'role': 'user', 'content': 'What is 6 times 7? Answer with just the number.'}],
    'max_tokens': 512, 'temperature': 0.3}).encode()
req = urllib.request.Request(f'{public}/v1/chat/completions', data=body,
    headers={'Authorization': f'Bearer {API_KEY}', 'Content-Type': 'application/json'})
print(json.loads(urllib.request.urlopen(req, timeout=120).read())['choices'][0]['message']['content'][-400:])

## Now, on your laptop

```powershell
$env:DATASET = "$env:USERPROFILE\models\reasoning\gsm-symbolic-p2.jsonl"
cargo run -p samaritan-run --example reason_eval
```

The harness now runs against the A100 model. Leave this notebook running.
The first real number from a capable substrate — it should beat the local
4B's 2/10 on p2.

## Optional: self-training fine-tune here too

Generate the verified set locally (`selftrain_export`), upload it, and train
on this same A100. Uploads `reasoning-selftrain.jsonl` and
`qdora_deviant.py` from your machine (or clone your repo), then runs it.

In [ ]:
# from google.colab import files; files.upload()   # upload reasoning-selftrain.jsonl + qdora_deviant.py + requirements.txt
# !pip -q install -r requirements.txt
# !python qdora_deviant.py reasoning-selftrain.jsonl --base-model Qwen/Qwen3-4B-Thinking-2507 --allow-small
# then download the adapter and convert to GGUF (training/README.md)
print('uncomment the lines above to train; see training/README.md')

## Shutdown (run when done)

In [ ]:
for p in ['vllm', 'cf']:
    try: globals()[p].terminate()
    except Exception: pass
print('stopped vLLM and the tunnel. Also: Runtime -> Disconnect and delete runtime.')